In [0]:
%sql
CREATE DATABASE IF NOT EXISTS flight_tracker
COMMENT 'Flight Price Tracker pipeline tables'

In [0]:
%sql
SHOW CATALOGS


catalog
samples
system
workspace


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.flight_tracker;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.flight_tracker.routes_config (
    route_id              STRING,
    origin                STRING,
    destination           STRING,
    origin_city           STRING,
    destination_city      STRING,
    origin_country        STRING,
    destination_country   STRING,
    route_type            STRING,
    currency              STRING,
    is_active             BOOLEAN,
    priority              INT,
    added_by              STRING,
    created_at            TIMESTAMP,
    updated_at            TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed'       = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.flight_tracker.flight_prices_raw (
    run_id                STRING,
    route_id              STRING,
    origin                STRING,
    destination           STRING,
    airline               STRING,
    airline_code          STRING,
    aircraft_type         STRING,
    flight_number         STRING,
    num_stops             INT,
    stop_locations        STRING,
    departure_datetime    TIMESTAMP,
    arrival_datetime      TIMESTAMP,
    duration_minutes      INT,
    price_usd             DOUBLE,
    cabin_class           STRING,
    seats_available       INT,
    baggage_allowance_kg  INT,
    is_refundable         BOOLEAN,
    source_api            STRING,
    api_response_code     INT,
    api_version           STRING,
    raw_payload           STRING,
    ingested_at           TIMESTAMP,
    record_checksum       STRING
)
USING DELTA
PARTITIONED BY (route_id)
TBLPROPERTIES (
    'delta.enableChangeDataFeed'           = 'true',
    'delta.autoOptimize.optimizeWrite'     = 'true',
    'delta.autoOptimize.autoCompact'       = 'true',
    'delta.logRetentionDuration'           = 'interval 90 days',
    'delta.deletedFileRetentionDuration'   = 'interval 90 days'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.flight_tracker.flight_prices_clean (
    run_id                STRING,
    route_id              STRING,
    origin                STRING,
    destination           STRING,
    airline               STRING,
    airline_code          STRING,
    aircraft_type         STRING,
    flight_number         STRING,
    num_stops             INT,
    stop_locations        STRING,
    departure_datetime    TIMESTAMP,
    arrival_datetime      TIMESTAMP,
    duration_minutes      INT,
    price_usd             DOUBLE,
    cabin_class           STRING,
    seats_available       INT,
    baggage_allowance_kg  INT,
    is_refundable         BOOLEAN,
    ingested_at           TIMESTAMP,
    record_checksum       STRING,
    is_valid              BOOLEAN,
    invalid_reason        STRING,
    is_duplicate          BOOLEAN,
    dq_check_version      STRING,
    transformed_at        TIMESTAMP
)
USING DELTA
PARTITIONED BY (route_id)
TBLPROPERTIES (
    'delta.enableChangeDataFeed'           = 'true',
    'delta.autoOptimize.optimizeWrite'     = 'true',
    'delta.autoOptimize.autoCompact'       = 'true',
    'delta.logRetentionDuration'           = 'interval 90 days',
    'delta.deletedFileRetentionDuration'   = 'interval 90 days'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.flight_tracker.flight_prices_avg (
    route_id              STRING,
    airline               STRING,
    airline_code          STRING,
    cabin_class           STRING,
    avg_7day_price        DOUBLE,
    min_7day_price        DOUBLE,
    max_7day_price        DOUBLE,
    stddev_7day_price     DOUBLE,
    latest_price          DOUBLE,
    price_drop_pct        DOUBLE,
    price_trend           STRING,
    data_points_count     INT,
    calculated_at         TIMESTAMP,
    valid_from            TIMESTAMP,
    valid_to              TIMESTAMP,
    is_current            BOOLEAN
)
USING DELTA
PARTITIONED BY (route_id)
TBLPROPERTIES (
    'delta.enableChangeDataFeed'           = 'true',
    'delta.autoOptimize.optimizeWrite'     = 'true',
    'delta.autoOptimize.autoCompact'       = 'true'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.flight_tracker.flight_price_alerts (
    alert_id              STRING,
    run_id                STRING,
    route_id              STRING,
    airline               STRING,
    airline_code          STRING,
    cabin_class           STRING,
    current_price         DOUBLE,
    avg_7day_price        DOUBLE,
    min_7day_price        DOUBLE,
    drop_pct              DOUBLE,
    price_trend           STRING,
    departure_datetime    TIMESTAMP,
    seats_available       INT,
    alert_type            STRING,
    alert_severity        STRING,
    email_recipient       STRING,
    email_subject         STRING,
    email_status          STRING,
    email_error_message   STRING,
    alert_sent_at         TIMESTAMP,
    alert_created_at      TIMESTAMP
)
USING DELTA
PARTITIONED BY (route_id)
TBLPROPERTIES (
    'delta.enableChangeDataFeed'           = 'true',
    'delta.autoOptimize.optimizeWrite'     = 'true',
    'delta.autoOptimize.autoCompact'       = 'true'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.flight_tracker.pipeline_audit_log (
    run_id                STRING,
    job_name              STRING,
    notebook_name         STRING,
    task_sequence         INT,
    cluster_id            STRING,
    start_time            TIMESTAMP,
    end_time              TIMESTAMP,
    duration_seconds      INT,
    status                STRING,
    records_read          INT,
    records_written       INT,
    records_skipped       INT,
    records_failed        INT,
    error_message         STRING,
    error_stacktrace      STRING,
    retry_attempt         INT,
    triggered_by          STRING,
    databricks_job_run_id STRING,
    created_at            TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.flight_tracker.api_error_log (
    error_id              STRING,
    run_id                STRING,
    route_id              STRING,
    api_endpoint          STRING,
    http_status_code      INT,
    error_code            STRING,
    error_message         STRING,
    request_payload       STRING,
    response_payload      STRING,
    retry_attempt         INT,
    max_retries           INT,
    resolved              BOOLEAN,
    resolved_at           TIMESTAMP,
    failed_at             TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
);

In [0]:
%sql
INSERT INTO workspace.flight_tracker.routes_config VALUES
('DEL-BOM', 'DEL', 'BOM', 'Delhi',   'Mumbai',    'India', 'India',     'DOMESTIC',      'USD', true, 1, 'system', current_timestamp(), current_timestamp()),
('DEL-BLR', 'DEL', 'BLR', 'Delhi',   'Bangalore', 'India', 'India',     'DOMESTIC',      'USD', true, 1, 'system', current_timestamp(), current_timestamp()),
('BOM-MAA', 'BOM', 'MAA', 'Mumbai',  'Chennai',   'India', 'India',     'DOMESTIC',      'USD', true, 2, 'system', current_timestamp(), current_timestamp()),
('CCU-DEL', 'CCU', 'DEL', 'Kolkata', 'Delhi',     'India', 'India',     'DOMESTIC',      'USD', true, 2, 'system', current_timestamp(), current_timestamp()),
('DEL-DXB', 'DEL', 'DXB', 'Delhi',   'Dubai',     'India', 'UAE',       'INTERNATIONAL', 'USD', true, 1, 'system', current_timestamp(), current_timestamp()),
('BOM-LHR', 'BOM', 'LHR', 'Mumbai',  'London',    'India', 'UK',        'INTERNATIONAL', 'USD', true, 1, 'system', current_timestamp(), current_timestamp()),
('DEL-SIN', 'DEL', 'SIN', 'Delhi',   'Singapore', 'India', 'Singapore', 'INTERNATIONAL', 'USD', true, 2, 'system', current_timestamp(), current_timestamp()),
('BOM-JFK', 'BOM', 'JFK', 'Mumbai',  'New York',  'India', 'USA',       'INTERNATIONAL', 'USD', true, 2, 'system', current_timestamp(), current_timestamp());

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
SHOW TABLES IN workspace.flight_tracker;

database,tableName,isTemporary
flight_tracker,api_error_log,false
flight_tracker,flight_price_alerts,false
flight_tracker,flight_prices_avg,false
flight_tracker,flight_prices_clean,false
flight_tracker,flight_prices_raw,false
flight_tracker,pipeline_audit_log,false
flight_tracker,routes_config,false


In [0]:
%sql
SELECT * FROM workspace.flight_tracker.routes_config;

route_id,origin,destination,origin_city,destination_city,origin_country,destination_country,route_type,currency,is_active,priority,added_by,created_at,updated_at
DEL-BOM,DEL,BOM,Delhi,Mumbai,India,India,DOMESTIC,USD,true,1,system,2026-06-29T11:28:56.932Z,2026-06-29T11:28:56.932Z
DEL-BLR,DEL,BLR,Delhi,Bangalore,India,India,DOMESTIC,USD,true,1,system,2026-06-29T11:28:56.932Z,2026-06-29T11:28:56.932Z
BOM-MAA,BOM,MAA,Mumbai,Chennai,India,India,DOMESTIC,USD,true,2,system,2026-06-29T11:28:56.932Z,2026-06-29T11:28:56.932Z
CCU-DEL,CCU,DEL,Kolkata,Delhi,India,India,DOMESTIC,USD,true,2,system,2026-06-29T11:28:56.932Z,2026-06-29T11:28:56.932Z
DEL-DXB,DEL,DXB,Delhi,Dubai,India,UAE,INTERNATIONAL,USD,true,1,system,2026-06-29T11:28:56.932Z,2026-06-29T11:28:56.932Z
BOM-LHR,BOM,LHR,Mumbai,London,India,UK,INTERNATIONAL,USD,true,1,system,2026-06-29T11:28:56.932Z,2026-06-29T11:28:56.932Z
DEL-SIN,DEL,SIN,Delhi,Singapore,India,Singapore,INTERNATIONAL,USD,true,2,system,2026-06-29T11:28:56.932Z,2026-06-29T11:28:56.932Z
BOM-JFK,BOM,JFK,Mumbai,New York,India,USA,INTERNATIONAL,USD,true,2,system,2026-06-29T11:28:56.932Z,2026-06-29T11:28:56.932Z
